In [2]:
import pandas as pd

In [3]:
subscribers = pd.read_csv("subscribers.csv")

In [4]:
subscribers["signup_date"] = pd.to_datetime(subscribers["signup_date"], format="%d-%m-%Y")
subscribers["churn_date"] = pd.to_datetime(subscribers["churn_date"], format="%d-%m-%Y")

In [7]:
cohort_data = subscribers[["subscriber_id", "signup_date", "churn_date"]].copy()
cohort_data["cohort_date"] = (cohort_data["signup_date"].dt.to_period("M"))

cohort_data.head()

,subscriber_id,signup_date,churn_date,cohort_date
0,SUB100000,2023-01-09,NaT,2023-01
1,SUB100001,2020-02-05,2022-12-08,2020-02
2,SUB100002,2022-04-15,NaT,2022-04
3,SUB100003,2024-10-10,NaT,2024-10
4,SUB100004,2020-04-12,2021-12-15,2020-04


In [8]:
today = pd.Timestamp.today()
cohort_data["effective_churn_date"] = (cohort_data["churn_date"].fillna(today))
cohort_data = cohort_data[["subscriber_id", "signup_date", "cohort_date", "effective_churn_date"]]
cohort_data.head()

,subscriber_id,signup_date,cohort_date,effective_churn_date
0,SUB100000,2023-01-09,2023-01,2026-09-21 16:37:14.681031
1,SUB100001,2020-02-05,2020-02,2022-12-08 00:00:00.000000
2,SUB100002,2022-04-15,2022-04,2026-09-21 16:37:14.681031
3,SUB100003,2024-10-10,2024-10,2026-09-21 16:37:14.681031
4,SUB100004,2020-04-12,2020-04,2021-12-15 00:00:00.000000


In [9]:
cohort_data["lifetime_months"] = ((cohort_data["effective_churn_date"] - cohort_data["signup_date"]).dt.days / 30).astype(int)

cohort_data[["subscriber_id","signup_date","cohort_date","effective_churn_date","lifetime_months"]]
cohort_data.head()

,subscriber_id,signup_date,cohort_date,effective_churn_date,lifetime_months
0,SUB100000,2023-01-09,2023-01,2026-09-21 16:37:14.681031,45
1,SUB100001,2020-02-05,2020-02,2022-12-08 00:00:00.000000,34
2,SUB100002,2022-04-15,2022-04,2026-09-21 16:37:14.681031,54
3,SUB100003,2024-10-10,2024-10,2026-09-21 16:37:14.681031,23
4,SUB100004,2020-04-12,2020-04,2021-12-15 00:00:00.000000,20


In [10]:
cohort_data["Active_3M"] = (cohort_data["lifetime_months"] >= 3).astype(int)

In [11]:
cohort_data["Active_6M"] = (cohort_data["lifetime_months"] >= 6).astype(int)

In [12]:
cohort_data["Active_12M"] = (cohort_data["lifetime_months"] >= 12).astype(int)

In [13]:
cohort_data[["subscriber_id","lifetime_months","Active_3M","Active_6M","Active_12M"]].head()

,subscriber_id,lifetime_months,Active_3M,Active_6M,Active_12M
0,SUB100000,45,1,1,1
1,SUB100001,34,1,1,1
2,SUB100002,54,1,1,1
3,SUB100003,23,1,1,1
4,SUB100004,20,1,1,1


In [14]:
cohort_retention = (cohort_data.pivot_table(index="cohort_date",values=["Active_3M","Active_6M","Active_12M"],aggfunc="mean")*100).round(2)
cohort_retention

,Active_12M,Active_3M,Active_6M
cohort_date,,,
2016-01,75.00,75.00,75.00
2016-02,88.89,88.89,88.89
2016-03,100.00,100.00,100.00
2016-04,100.00,100.00,100.00
2016-05,85.00,100.00,90.00
...,...,...,...
2026-01,0.00,89.22,83.19
2026-02,0.00,86.11,80.56
2026-03,0.00,79.44,61.68


### Cohort Retention Observations

- Retention generally decreases as the subscriber lifecycle increases, with **12-month retention lower than 3-month retention** for mature cohorts.
- Recent cohorts show **0% or unavailable 12-month retention because they have not yet reached the 12-month observation period**, so they should not be interpreted as actual churn.
- The cohort table helps compare **retention patterns across different signup periods** and identify changes in subscriber retention over time.